In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains.question_answering import load_qa_chain


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [3]:
# Load dos modelos (Embedding e LLM)

embedding_model = OpenAIEmbeddings()
llm = ChatOpenAI(model_name = "gpt-3.5-turbo", max_tokens = 200)

In [4]:
# Carregar PDF

pdf_link = "Clean Architecture.pdf"
loader = PyPDFLoader(pdf_link, extract_images=False)

pages = loader.load_and_split()
len(pages)

381

In [5]:
# Separar em Chunks (pedaços de documento)

text_spliter = RecursiveCharacterTextSplitter(
    chunk_size = 4000, # padrão que normalmente é utlizado no mercado
    chunk_overlap = 20, # evita perda de conteúdo no meio do documento e permite manter o contexto nos pedaços de documento
    length_function = len,
    add_start_index = True
)

chunks = text_spliter.split_documents(pages)
len(chunks)

381

In [6]:
# Salvar no Vector DB - Chroma
db = Chroma.from_documents(chunks, embedding=embedding_model, persist_directory="db/text_index")
db.persist()

/tmp/ipykernel_13963/2634103074.py:3: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


In [7]:
# Carregar DB
vectordb = Chroma(persist_directory="db/text_index", embedding_function=embedding_model)

# Loader Retriever - busca os 3 primeiros documentos relevantes
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

# Construção da cadeia de prompt para chamada LLM
chain = load_qa_chain(llm, chain_type="stuff")

/tmp/ipykernel_13963/543418372.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectordb = Chroma(persist_directory="db/text_index", embedding_function=embedding_model)
/tmp/ipykernel_13963/543418372.py:8: LangChainDeprecationWarning: This class is deprecated. See the following migration guides for replacements based on `chain_type`:
stuff: https://python.langchain.com/docs/versions/migrating_chains/stuff_docs_chain
map_reduce: https://python.langchain.com/docs/versions/migrating_chains/map_reduce_chain
refine: https://python.langchain.com/docs/versions/migrating_chains/refine_chain
map_rerank: https://python.langchain.com/docs/versions/migrating_chains/map_rerank_docs_chain

See also guides on retrieval and question-

In [8]:
def ask(question):
    context = retriever.invoke(question)
    answer = (chain({"input_documents": context, "question": question}, return_only_outputs=True))['output_text']
    return answer, context
    

In [10]:
user_question = input("User: ")
answer, context = ask(user_question)
print("Answer: ", answer)
print("Context: ", context)

Answer:  A principal diferença entre a Clean Architecture e a Hexagonal Architecture reside na forma como os componentes do sistema são organizados. Ambas as arquiteturas têm o objetivo de separar as preocupações e manter uma estrutura limpa e organizada, no entanto, a Clean Architecture prioriza a separação de camadas em níveis de abstração, garantindo que os componentes de alto nível não dependam dos de baixo nível. Por outro lado, a Hexagonal Architecture, também conhecida como Ports and Adapters, enfatiza a separação de módulos de aplicação do ambiente externo, como bancos de dados e interfaces de usuário, para permitir que o sistema seja mais testável e adaptável a mudanças.
Context:  [Document(metadata={'page_label': '195', 'producer': 'calibre 3.7.0 [https://calibre-ebook.com]', 'creationdate': '2017-09-15T12:39:59+00:00', 'author': 'Robert C. Martin', 'total_pages': 429, 'page': 194, 'pxcviewerinfo': "PDF-XChange Viewer;2.5.313.1;Jun  8 2015;11:48:33;D:20170920170036+08'00'", '